In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("merged_transactions_feedback.csv")

customer_df = df.groupby("Customer_ID").agg({
    "Transaction_Amount": "sum",
    "Satisfaction_Score": "mean",
    "Likelihood_to_Recommend": "mean",
    "Transaction_ID": "count"
}).reset_index()

customer_df.rename(columns={
    "Transaction_Amount": "Total_Transaction_Amount",
    "Transaction_ID": "Transaction_Count",
    "Satisfaction_Score": "Avg_Satisfaction_Score",
    "Likelihood_to_Recommend": "Avg_Likelihood_to_Recommend"
}, inplace=True)

customer_df["Log_Total_Transaction_Amount"] = np.log1p(customer_df["Total_Transaction_Amount"])

customer_df.head()

,Customer_ID,Total_Transaction_Amount,Avg_Satisfaction_Score,Avg_Likelihood_to_Recommend,Transaction_Count,Log_Total_Transaction_Amount
0,1,33672.0,8.500000,9.000000,12,10.424452
1,2,14721.0,4.333333,5.000000,6,9.597098
2,3,4614.0,8.333333,3.666667,3,8.437067
3,4,49770.0,5.500000,3.333333,12,10.815188
4,5,133182.0,6.444444,5.666667,45,11.799479


In [2]:
p33 = customer_df["Total_Transaction_Amount"].quantile(0.33)
p66 = customer_df["Total_Transaction_Amount"].quantile(0.66)

def income_group(x):
    if x <= p33:
        return "Low"
    elif x <= p66:
        return "Middle"
    else:
        return "High"

customer_df["Income_Level_Group"] = customer_df["Total_Transaction_Amount"].apply(income_group)

customer_df["Income_Level_Group"].value_counts()

Income_Level_Group
High      338
Low       328
Middle    327
Name: count, dtype: int64

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = [
    "Log_Total_Transaction_Amount",   # use LOG here
    "Avg_Satisfaction_Score",
    "Avg_Likelihood_to_Recommend",
    "Transaction_Count"
]

X = customer_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
customer_df["Cluster"] = kmeans.fit_predict(X_scaled)

pd.crosstab(customer_df["Cluster"], customer_df["Income_Level_Group"])

In [4]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
customer_df["Cluster"] = kmeans.fit_predict(X_scaled)

customer_df.head()

/lib/python3.13/site-packages/threadpoolctl.py:1123: RuntimeWarning: JsProxy.as_object_map() is deprecated. Use as_py_json() instead.
  for filepath in LDSO.loadedLibsByName.as_object_map():


,Customer_ID,Total_Transaction_Amount,Avg_Satisfaction_Score,Avg_Likelihood_to_Recommend,Transaction_Count,Log_Total_Transaction_Amount,Income_Level_Group,Cluster
0,1,33672.0,8.500000,9.000000,12,10.424452,Low,0
1,2,14721.0,4.333333,5.000000,6,9.597098,Low,0
2,3,4614.0,8.333333,3.666667,3,8.437067,Low,1
3,4,49770.0,5.500000,3.333333,12,10.815188,Middle,1
4,5,133182.0,6.444444,5.666667,45,11.799479,High,1


In [5]:
pd.crosstab(customer_df["Cluster"], customer_df["Income_Level_Group"])

Income_Level_Group,High,Low,Middle
Cluster,,,
0,90,235,200
1,241,93,127
2,7,0,0
